# DQN Dueling - Entrenamiento para FFAI Assistant

Arquitectura Dueling DQN para 15 acciones.

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow import keras
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Dueling DQN Architecture
class DuelingDQN(keras.Model):
    def __init__(self, state_size=256, action_size=15):
        super().__init__()
        self.fc1 = keras.layers.Dense(256, activation='relu')
        self.fc2 = keras.layers.Dense(128, activation='relu')
        self.value_fc = keras.layers.Dense(64, activation='relu')
        self.value_out = keras.layers.Dense(1)
        self.adv_fc = keras.layers.Dense(64, activation='relu')
        self.adv_out = keras.layers.Dense(action_size)
    
    def call(self, x):
        x = self.fc1(x)
        x = self.fc2(x)
        value = self.value_out(self.value_fc(x))
        advantage = self.adv_out(self.adv_fc(x))
        return value + (advantage - tf.reduce_mean(advantage, axis=1, keepdims=True))

# Crear modelos
STATE_SIZE, ACTION_SIZE = 256, 15
policy_net = DuelingDQN(STATE_SIZE, ACTION_SIZE)
target_net = DuelingDQN(STATE_SIZE, ACTION_SIZE)
policy_net.build(input_shape=(None, STATE_SIZE))
target_net.build(input_shape=(None, STATE_SIZE))
target_net.set_weights(policy_net.get_weights())
print(f"✓ Modelos creados: {policy_net.count_params():,} params")

In [ ]:
# Exportar a TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(policy_net)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('dqn_dueling.tflite', 'wb') as f:
    f.write(tflite_model)

# Target network
converter2 = tf.lite.TFLiteConverter.from_keras_model(target_net)
converter2.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_target = converter2.convert()

with open('dqn_target.tflite', 'wb') as f:
    f.write(tflite_target)

print("✓ Modelos exportados: dqn_dueling.tflite, dqn_target.tflite")

In [ ]:
# Descargar
from google.colab import files
files.download('dqn_dueling.tflite')
files.download('dqn_target.tflite')